# Lesson 02 - חקירת מסגרת Microsoft Agent

ה-**Microsoft Agent Framework (MAF)** הוא מסגרת מאוחדת לבניית סוכני בינה מלאכותית. הוא מספק ארכיטקטורה ברורה, ניתנת להרכבה עם ארבעה בלוקים מרכזיים:

- **לקוח** – מתחבר לנקודת קצה של מודל בינה מלאכותית ומטפל בתקשורת
- **סוכן** – עוטף לקוח עם הוראות והגדרות כלים
- **כלים** – מרחיבים את יכולות הסוכן עם פונקציות מותאמות שהמודל יכול לקרוא להן
- **מפגש** – שומר היסטוריית שיחה עבור אינטראקציות מרובות סיבובים

בשיעור זה, נבנה **סוכן הזמנת נסיעות** שבודק זמינות יעד באמצעות המושגים הללו.


## הגדרה


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## הבנת ארכיטקטורת מסגרת הסוכן

מסגרת הסוכן של מייקרוסופט פועלת על פי ארכיטקטורה רב-שכבתית:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **לקוח** – `AzureAIProjectAgentProvider` מתחבר לפריסת Azure OpenAI. הוא מטפל באימות, עיצוב הבקשות ופירוש התגובות.
2. **סוכן** – נוצר מהלקוח דרך `provider.create_agent()`, הסוכן משלב גישה למודל עם הוראות (פרומפט מערכת) וכלים.
3. **כלים** – פונקציות פייתון המעוטרות עם `@tool` שהסוכן יכול להפעיל לבצע פעולות או לשאוב נתונים.
4. **מפגש** – אובייקט `AgentSession` (נוצר דרך `agent.create_session()`) ששומר היסטוריית שיחה, ומאפשר דיאלוג רב-סיבובים שבו הסוכן זוכר הקשר קודם.

בוא נבנה כל שכבה שלב אחר שלב.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## הוספת כלים עם המחלק @tool

כלים מאפשרים לסוכנים לבצע פעולות מעבר ליצירת טקסט. המחלק `@tool` הופך פונקציית פייתון רגילה למשהו שהסוכן יכול לקרוא לו.

נקודות מפתח:  
- השתמש ב-`Annotated[type, "description"]` כדי שהמודל יבין כל פרמטר.  
- מחרוזת התיעוד הופכת לתיאור הכלי שהמודל רואה.  
- `approval_mode="never_require"` אומר שהכלי מופעל אוטומטית ללא אישור משתמש.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## יצירת סוכן עם כלים

כעת אנו משלבים את הלקוח, ההוראות, והכלים לסוכן. ה-`instructions` משמשות כהנחיית המערכת — הן מגדירות את האישיות וההתנהגות של הסוכן.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## שיחות מרובות תורות עם מושבים

`AgentSession` (נוצר באמצעות `agent.create_session()`) עוקב אחר כל ההודעות בשיחה. על ידי העברת אותו מושב לכל קריאה של `agent.run()`, הסוכן יכול לגשת להיסטוריית השיחה המלאה ולהתייחס להודעות קודמות.

אנו מעבירים `tools=[check_destination_availability]` כדי שהסוכן יוכל לקרוא לבודק הזמינות שלנו בכל תור.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## סיכום

בשיעור זה חקרתם את ארבעת העמודים של מסגרת ה-Microsoft Agent:

| מושג | מה שלמדתם |
|---------|------------------|
| **לקוח** | `AzureAIProjectAgentProvider` מתחבר ל-Azure OpenAI באמצעות אימות מבוסס הרשאות |
| **סוכן** | `provider.create_agent()` מארגן חיבור למודל עם הוראות ושם |
| **כלים** | הדקורטור `@tool` חושף פונקציות פייתון שקובע הסוכן יוכל לקרוא להן |
| **מפגש** | `agent.create_session()` שומר היסטוריית שיחה על פני מספר סבבים |

אבני הבניין האלה מתמזגות יחד כדי ליצור סוכנים שיכולים לנהל שיחות טבעיות, לקרוא לפונקציות חיצוניות ולשמר הקשר — הבסיס לתבניות סוכנות מתקדמות יותר בשיעורים בהמשך.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:  
מסמך זה תורגם באמצעות שירות תרגום מבוסס בינה מלאכותית [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל טעויות או אי-דיוקים. המסמך המקורי בשפת המקור שלו נחשב למקור הסמכותי. למידע קריטי מומלץ להשתמש בשירותי תרגום מקצועיים על ידי מתרגם אנושי. איננו נושאים באחריות לכל אי-הבנה או פרשנות שגויה הנובעת משימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
